# Data Flow Control (DFC / Passant) — smoke test

A minimal notebook to confirm the `data-flow-control` package is installed and working in this venv.

DFC is a **SQL query-rewrite engine**: you wrap a database connection, register **policies** over source tables, and DFC rewrites every query to enforce them before execution. We test the core behaviors:

1. Import + version
2. Set up a DuckDB table and wrap it with `dfc(...)`
3. **Passthrough** — no policy → unchanged results
4. **REMOVE** — policy filters out violating rows
5. **`explain()`** — DFC's data-flow analysis of a query
6. **KILL** — policy raises on a violating query
7. **Policy management** — `policies()` / `delete_policy()`

> If every cell runs without error and the assertions pass, DFC is healthy.

In [1]:
import data_flow_control as dfcmod
from data_flow_control import Policy, Resolution, dfc, Dialect, RewriteOptions, PassantRewriteError
import duckdb

print("data_flow_control imported OK")
print("duckdb version:", duckdb.__version__)
print("Resolution modes:", [r for r in dir(Resolution) if not r.startswith("_")])
print("Public API:", [s for s in dir(dfcmod) if not s.startswith("_") and s[0].isupper()])

data_flow_control imported OK
duckdb version: 1.5.3
Resolution modes: ['KILL', 'RELATION_UDF', 'REMOVE', 'UDF', 'UI']
Public API: ['Dialect', 'PassantRewriteError', 'Policy', 'Resolution', 'RewriteOptions', 'UiUpdateMode', 'UiViolationEvent']


## 1. Build a table and wrap the connection
A tiny `orders` table in an in-memory DuckDB. `dfc(raw)` returns a policy-enforcing wrapper around the raw connection.

In [2]:
raw = duckdb.connect()
raw.execute("CREATE TABLE orders (id INTEGER, region TEXT, amount INTEGER)")
raw.execute("INSERT INTO orders VALUES (1, 'us', 100), (2, 'eu', 200), (3, 'us', 300)")

conn = dfc(raw)
print("wrapped connection:", type(conn).__name__)

wrapped connection: Connection


## 2. Passthrough — no policy registered
With no policy, DFC should return the full, unmodified result.

In [3]:
rows = conn.fetchall("SELECT id, region, amount FROM orders ORDER BY id")
print("passthrough:", rows)
assert rows == [(1, 'us', 100), (2, 'eu', 200), (3, 'us', 300)]
print("PASS: passthrough returns all rows")

passthrough: [(1, 'us', 100), (2, 'eu', 200), (3, 'us', 300)]
PASS: passthrough returns all rows


## 3. REMOVE policy — filter violating rows
Register a policy: rows from `orders` must satisfy `region = 'us'`; on failure, **REMOVE** them. DFC rewrites the query so only `us` rows survive — without us changing the SQL.

In [4]:
policy = Policy(sources=["orders"], constraint="orders.region = 'us'", on_fail=Resolution.REMOVE)
conn.register_policy(policy)

rows = conn.fetchall("SELECT id, region, amount FROM orders ORDER BY id")
print("after REMOVE policy:", rows)
assert rows == [(1, 'us', 100), (3, 'us', 300)], "eu row should have been filtered"
print("PASS: REMOVE filtered the 'eu' row deterministically")

after REMOVE policy: [(1, 'us', 100), (3, 'us', 300)]
PASS: REMOVE filtered the 'eu' row deterministically


## 4. `explain()` — DFC's data-flow analysis
DFC analyzes which source tables flow into a query and how. `explain()` exposes that scope — evidence the rewrite is derivation-aware, not a string hack.

In [5]:
info = conn.explain("SELECT id, amount FROM orders WHERE amount > 50")
scope = info.get("scope", info) if isinstance(info, dict) else info
print("visible_tables:", scope.get("visible_tables"))
print("is_aggregation:", scope.get("is_aggregation"), "| has_join:", scope.get("has_join"))
print("PASS: explain() returned data-flow scope")

visible_tables: ['orders']
is_aggregation: False | has_join: False
PASS: explain() returned data-flow scope


## 5. KILL policy — raise on violation
On a fresh connection, use **KILL**: instead of silently filtering, a query that would touch a violating row raises an error. This is the hard-stop mode.

In [6]:
raw2 = duckdb.connect()
raw2.execute("CREATE TABLE t (id INTEGER, region TEXT)")
raw2.execute("INSERT INTO t VALUES (1, 'us'), (2, 'eu')")
c2 = dfc(raw2)
c2.register_policy(Policy(sources=["t"], constraint="t.region = 'us'", on_fail=Resolution.KILL))

raised = False
try:
    c2.fetchall("SELECT id, region FROM t")
except Exception as e:
    raised = True
    msg = str(e)
    print("raised:", type(e).__name__)
    print("message contains policy violation:", "dfc policy violation" in msg or "KILL" in msg)
assert raised, "KILL policy should have raised on the violating 'eu' row"
print("PASS: KILL raised on violation")

raised: InvalidInputException
message contains policy violation: True
PASS: KILL raised on violation


## 6. Policy management
`policies()` lists registered policies; `delete_policy()` removes them and restores passthrough.

In [7]:
print("registered policies:", len(conn.policies()))
for p in conn.policies():
    print("  -", p.sources, "|", p.constraint, "|", p.on_fail)

conn.delete_policy(sources=["orders"])  # delete_policy matches on sources, not the Policy object
rows = conn.fetchall("SELECT id FROM orders ORDER BY id")
print("after delete_policy:", rows)
assert rows == [(1,), (2,), (3,)], "deleting the policy should restore all rows"
print("PASS: delete_policy restored passthrough")

registered policies: 1
  - ['orders'] | orders.region = 'us' | Resolution.REMOVE
after delete_policy: [(1,), (2,), (3,)]
PASS: delete_policy restored passthrough


## Result

If you reached here with every `PASS` printed and no exceptions, **DFC is installed and working correctly** — import, connection wrapping, REMOVE, explain, KILL, and policy management all function.

Next step (separate discussion): wiring DFC in as an **AgentDyn defense** so we can measure its effect on the prompt-injection benchmark.

In [8]:
print("\n==== DFC SMOKE TEST COMPLETE — all checks passed ====")


==== DFC SMOKE TEST COMPLETE — all checks passed ====
